# How the backward ray-casting works — step by step

This notebook opens up `mask_oblique` and follows **one pixel at a time** through
the whole process, using the real data types: the **oblique** photo, the **DSM**
(height map), the **ortho** (flat map), a **point**, and the **parcel/building
polygon**. It uses only the existing `vexcel_masker` functions plus a little
plotting.

The idea in one line: *for each pixel, trace the ray of light back out into the
world, find the first surface it hits using the DSM, and keep the pixel only if
that spot is inside the polygon.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shapely
from pyproj import Transformer
from shapely.ops import transform as shp_transform

import vexcel_masker as vm

token = vm.get_token_from_config("config.ini")   # or: token = "PASTE_A_TOKEN"
lon, lat = -76.876653, 38.757577                  # a house near Upper Marlboro, MD
print("logged in")

## Step 0 — the ingredients

We pull the five pieces for this spot, all from the same dated flight.

In [ ]:
coll   = vm.best_collection(token, lon, lat)
parcel = vm.get_parcel(token, lon, lat)          # polygon (property boundary), lon/lat
structs= vm.get_structures(token, lon, lat)      # polygons (building footprints)
dsm    = vm.get_dsm(token, lon, lat, collection=coll)     # height map (metres), in UTM
ortho  = vm.get_ortho(token, lon, lat, collection=coll)   # flat top-down map
north  = vm.get_oriented(token, lon, lat, "north", collection=coll)  # the oblique + camera

WORK = vm.working_crs(lon, lat)                  # the metres CRS everything lines up in
print("DSM", dsm.data.shape, dsm.crs, "| ortho", ortho.image.shape, "| oblique", north.image.shape)

### See the four data types

- **Oblique**: the angled photo we want to mask.
- **Ortho**: the same place flattened onto the ground (a map).
- **DSM**: a height for every ground location (brighter = taller — you can pick out roofs/trees).
- **Polygon on the ortho**: the parcel (red) and buildings (cyan) laid on the flat map.

In [ ]:
def draw_polys_on_ortho(ax, polys, color):
    tf = Transformer.from_crs("EPSG:4326", ortho.crs, always_xy=True)
    inv = ~ortho.transform
    for g in ([polys] if hasattr(polys, "exterior") else list(getattr(polys, "geoms", polys))):
        xs, ys = g.exterior.coords.xy
        X, Y = tf.transform(np.array(xs), np.array(ys))
        col = inv.a * X + inv.b * Y + inv.c
        row = inv.d * X + inv.e * Y + inv.f
        ax.plot(col, row, color=color, lw=2)

fig, ax = plt.subplots(2, 2, figsize=(12, 12))
ax[0,0].imshow(north.image);        ax[0,0].set_title("OBLIQUE (angled photo)")
ax[0,1].imshow(ortho.image);        ax[0,1].set_title("ORTHO (flattened map)")
im = ax[1,0].imshow(dsm.data, cmap="terrain"); ax[1,0].set_title("DSM (height map, metres)")
fig.colorbar(im, ax=ax[1,0], shrink=0.7)
ax[1,1].imshow(ortho.image);        ax[1,1].set_title("POLYGONS on the ortho")
draw_polys_on_ortho(ax[1,1], parcel, "red")
for s in structs: draw_polys_on_ortho(ax[1,1], s, "cyan")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## Step 1 — pick a pixel

To make it concrete we choose two known spots on the ground and see where they
land in the oblique photo:

- **G — a yard point** (on the ground, inside the parcel)
- **R — the house roof** (elevated ~ the roof height from the DSM)

We use the camera to project each world point to a pixel (`world_to_pixel`).

In [ ]:
to_utm = Transformer.from_crs("EPSG:4326", WORK, always_xy=True)

# house = the biggest building footprint; its centroid, in metres
house = max(structs, key=lambda p: p.area)
hx, hy = to_utm.transform(house.centroid.x, house.centroid.y)
roof_z = float(np.atleast_1d(dsm.sample(np.array([hx]), np.array([hy])))[0])   # roof height

# a yard point ~15 m east of the house, at ground height
gx, gy = hx + 15.0, hy
grnd_z = float(np.atleast_1d(dsm.sample(np.array([gx]), np.array([gy])))[0])

R_world = (hx, hy, roof_z)     # the roof
G_world = (gx, gy, grnd_z)     # the yard
print(f"roof height ~ {roof_z:.1f} m,  ground ~ {grnd_z:.1f} m  (difference = the building)")

uG, vG = [float(x) for x in north.camera.world_to_pixel(*G_world)]
uR, vR = [float(x) for x in north.camera.world_to_pixel(*R_world)]

plt.figure(figsize=(8, 8)); plt.imshow(north.image)
plt.scatter([uG], [vG], c="yellow", s=120, ec="k", label="G — yard")
plt.scatter([uR], [vR], c="magenta", s=120, ec="k", label="R — roof")
plt.legend(); plt.title("Step 1: two pixels we'll trace"); plt.axis("off"); plt.show()

## Step 2 — turn the pixel into a ray

`pixel_to_ray` converts pixel **G** into a 3-D line: an **origin** (the camera,
way up in the air) and a **direction** (a unit vector pointing out through that
pixel). Every point on that line is a candidate for what the pixel is seeing.

In [ ]:
origin, direction = north.camera.pixel_to_ray(uG, vG)
print("camera / ray origin (X, Y, Z):", np.round(origin, 1), " (metres, ~how high the plane was)")
print("ray direction (unit vector):  ", np.round(direction, 3))

## Step 3 — march the ray and watch it meet the ground

Now we step along the ray. At each step we compare two heights:

- the **ray's** height as it descends, and
- the **DSM surface** height at the ray's current (X, Y).

While the ray is *above* the surface it keeps going; the moment it drops to the
surface is the **first hit** — the real spot the pixel sees. The plot below is a
side view of exactly that.

In [ ]:
# find the hit (the library does the same marching internally) to frame the plot
hit, found = dsm.first_hit(origin[None, :], direction[None, :])
s_hit = float(np.linalg.norm(hit[0] - origin))

s = np.linspace(0, s_hit * 1.12, 500)          # distance travelled along the ray (metres)
P = origin + s[:, None] * direction            # points along the ray
ray_height = P[:, 2]
surf_height = dsm.sample(P[:, 0], P[:, 1])      # DSM height beneath the ray

plt.figure(figsize=(11, 5))
plt.plot(s, ray_height, label="ray height (coming down)", color="tab:blue")
plt.plot(s, surf_height, label="DSM surface height", color="tab:green")
plt.axvline(s_hit, color="red", ls="--")
plt.scatter([s_hit], [hit[0, 2]], color="red", zorder=5, label="FIRST HIT")
plt.xlabel("distance along the ray from the camera (m)")
plt.ylabel("height above sea level (m)")
plt.title("Step 3: the ray descends until it meets the DSM surface")
plt.legend(); plt.grid(alpha=.3); plt.show()

## Step 4 — the first hit is the real point the pixel sees

Because we started from the yard point **G** and traced its pixel back out, the
hit should land right back on **G** — a nice round-trip that shows the camera and
DSM agree.

In [ ]:
print("we started from G =", np.round(G_world, 1))
print("ray's first hit  =", np.round(hit[0], 1))
print("match within", f"{np.linalg.norm(hit[0] - np.array(G_world)):.2f} m")

## Step 5 — drop the height, test the polygon

We keep only the **(X, Y)** of the hit (its ground footprint) and ask shapely
whether that point is inside the parcel. Inside → keep the pixel; outside → mask
it. That single true/false is the whole decision for this pixel.

In [ ]:
# parcel in the same metres CRS as the hit
parcel_m = shp_transform(lambda x, y, z=None: to_utm.transform(x, y), parcel)

X, Y = hit[0, 0], hit[0, 1]
inside = bool(shapely.contains_xy(parcel_m, X, Y))
print(f"hit (X, Y) = ({X:.1f}, {Y:.1f}) ->  inside parcel? {inside}  ->  {'KEEP' if inside else 'MASK'}")

fig, ax = plt.subplots(figsize=(7, 7)); ax.imshow(ortho.image)
draw_polys_on_ortho(ax, parcel, "red")
tf = Transformer.from_crs(WORK, ortho.crs, always_xy=True); inv = ~ortho.transform
ox, oy = tf.transform(X, Y); ax.scatter([inv.a*ox+inv.b*oy+inv.c], [inv.d*ox+inv.e*oy+inv.f],
                                        c="yellow", s=120, ec="k")
ax.set_title("Step 5: the hit point, on the ortho, vs the parcel"); ax.axis("off"); plt.show()

## Step 6 — do that for every pixel = the mask

`mask_oblique` runs Steps 1–5 for every pixel in the region. Below: the mask
(everything outside the parcel filled), and the outline (the boundary drawn on
the real photo).

In [ ]:
vm.show(vm.mask_oblique(north, dsm, parcel), "every pixel ray-cast, kept if inside the parcel")
vm.show(vm.outline_oblique(north, dsm, parcel), "the same boundary drawn on the photo")

## The 'aha' — why leaning neighbours are excluded (occlusion for free)

Trace the **roof** pixel R the same way. Its ray hits the roof *up in the air*,
not the ground — and we keep it only because that roof's (X, Y) is inside **this**
parcel. If R were a **neighbour's** roof leaning over the line, the first hit's
(X, Y) would fall **outside** the parcel, so it'd be masked — even though it
looks like it sits over your yard in the flat photo. Taking the **first** hit is
what makes that automatic.

In [ ]:
o2, d2 = north.camera.pixel_to_ray(uR, vR)
hit2, _ = dsm.first_hit(o2[None, :], d2[None, :])
s2 = float(np.linalg.norm(hit2[0] - o2)); ss = np.linspace(0, s2*1.12, 500)
P2 = o2 + ss[:, None] * d2
plt.figure(figsize=(11, 4))
plt.plot(ss, P2[:, 2], label="ray height", color="tab:blue")
plt.plot(ss, dsm.sample(P2[:, 0], P2[:, 1]), label="DSM surface", color="tab:green")
plt.scatter([s2], [hit2[0, 2]], color="red", zorder=5, label="FIRST HIT (on the ROOF, up high)")
plt.xlabel("distance along ray (m)"); plt.ylabel("height (m)")
plt.title("The roof pixel hits a surface ~"+f"{hit2[0,2]-grnd_z:.0f} m above the ground")
plt.legend(); plt.grid(alpha=.3); plt.show()
print("roof hit (X,Y) inside THIS parcel?", bool(shapely.contains_xy(parcel_m, hit2[0,0], hit2[0,1])),
      "  (a neighbour's roof would be False -> masked)")

## The ortho contrast — no ray needed

An ortho is already flattened onto the ground, so a pixel maps to a ground (X, Y)
by a single fixed lookup (the GeoTIFF's affine transform) — Steps 2–4 collapse to
one step. Then it's the **same** point-in-polygon test. That's why ortho masking
is simple, and obliques need the ray-cast.

In [ ]:
vm.show(vm.mask_ortho(ortho, parcel, fill="black"), "ortho: flat lookup + same polygon test")